# Phase 20A — Streamlit demo

Runs `app.py` (the Router-backed NL-to-query UI built on Mac) on a Colab GPU, where the trained checkpoints and Spider databases already live on Drive. The app itself is unchanged — this notebook just gets the same repo state a GPU session needs, then exposes the Streamlit port publicly via a `cloudflared` quick tunnel, since Colab doesn't let you hit `localhost:8501` directly.

Same checkpoint layout as `phase18_eval_ablation_res.ipynb` — if your Drive folder differs from `codegen/checkpoints/{sar,generator}_{sql,nosql}`, adjust the `DRIVE` path below.

## 0. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"GPU {i}: {name}  ({total_gb:.1f} GB)")
        # Mirrors the 20GiB cut in src/generator/infer.py -- keep the two in sync.
        if total_gb >= 20:
            print("  -> >=20GiB (A100/L4-class). GeneratorInfer loads full bf16, no quantization.")
        else:
            print("  -> <20GiB (T4-class). GeneratorInfer loads int8 via bitsandbytes.")
else:
    raise RuntimeError("No GPU visible -- set Runtime > Change runtime type > GPU before continuing.")

## 1. Clone repo + install dependencies

In [ ]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv pymongo bitsandbytes streamlit langgraph

## 2. Mount Drive and load checkpoints (both tracks)

Loads SAR + Generator checkpoints for **both** SQL and NoSQL from Drive. Same symlink trick as Phase 18's eval notebook, so `app.py`'s `configs/config.yaml` paths (`models/sar_sql`, `models/generator_sql`, ...) resolve without any code changes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)

for name in ['sar_sql', 'generator_sql', 'sar_nosql', 'generator_nosql']:
    dst = f'models/{name}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/checkpoints/{name}', dst)

!ls -la models/sar_sql models/generator_sql models/sar_nosql models/generator_nosql

In [ ]:
# Safety net: force sar.backend to memory. ChromaDB's PersistentClient can't open
# an index over a Google Drive FUSE mount, so this avoids that failure mode entirely.
text = open('configs/config.yaml').read()
text = text.replace('backend: chroma', 'backend: memory')
open('configs/config.yaml', 'w').write(text)
!grep -A1 "^sar:" configs/config.yaml | head -3

## 3. Spider SQLite databases

Needed for the SQL track to *execute* (not just generate) a query — `app.py`'s DB picker reads `Data/Spider/database/` the same way `scripts/run_baseline.py` does. Uploaded once as a zip to Drive (same artifact Phase 18 uses).

In [ ]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l

## 4. MongoDB setup (NoSQL track only)

Skip this section if you only plan to demo the SQL track. `Data/mongodb/*.json` schema-cache files are git-tracked from an earlier run on a different machine — `convert_all()` treats their existence as "already converted" and will silently skip real data insertion into this fresh session's empty `mongod`, so they're cleared first.

In [ ]:
!apt-get install -y mongodb >/dev/null 2>&1 || (curl -fsSL https://pgp.mongodb.com/server-7.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor && echo "deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list && apt-get update -qq && apt-get install -y mongodb-org)
!mkdir -p /data/db
import subprocess, time
subprocess.Popen(['mongod', '--dbpath', '/data/db', '--logpath', '/var/log/mongod.log', '--fork'])
time.sleep(3)
!tail -n 5 /var/log/mongod.log

In [ ]:
import shutil
shutil.rmtree('Data/mongodb', ignore_errors=True)   # force a real reconversion, see note above

from src.mongodb_converter import convert_all
convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",
)

## 5. DeepSeek API key

Needed by `SchemaLinker` (API mode) every time `app.py` routes a question. Paste your real key in place of the placeholder, then re-run this cell — it will not overwrite a key that's already there. **Never commit this cell with a real key filled in.**

In [ ]:
from pathlib import Path

PLACEHOLDER = 'your_actual_key_here'
env = Path('.env')
existing = env.read_text() if env.exists() else ''

if 'DEEPSEEK_API_KEY=' in existing and PLACEHOLDER not in existing:
    print('.env already contains a DEEPSEEK_API_KEY -- left untouched.')
else:
    env.write_text(f'DEEPSEEK_API_KEY={PLACEHOLDER}\n')
    print('Wrote .env with a placeholder. Edit it (or this cell) with your real key, then re-run.')

## 6. Launch Streamlit + expose it publicly

Colab has no direct access to `localhost:8501`, so a `cloudflared` quick tunnel proxies it to a public `*.trycloudflare.com` URL. Unlike `localtunnel`, it has no browser interstitial page in front of it — that interstitial is what was serving HTML in place of Streamlit's JS chunks and causing the `TypeError: Importing a module script failed` error, so this avoids that failure mode entirely.

The last cell runs in the foreground and stays busy on purpose (that's what keeps the tunnel alive) — use **Runtime > Interrupt execution** to stop the demo.

In [ ]:
import subprocess, socket, time

# Kill any previous run from an earlier attempt at this cell, so we don't end up
# with two streamlit processes fighting over the port.
subprocess.run(["pkill", "-f", "streamlit run app.py"], stderr=subprocess.DEVNULL)
time.sleep(1)

log = open('/content/streamlit.log', 'w')
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.headless", "true",
     "--server.address", "127.0.0.1", "--server.port", "8501"],
    stdout=log, stderr=subprocess.STDOUT,
)

# Poll the port instead of a fixed sleep -- model/import overhead can push
# startup past a fixed few-second wait, which is what left cloudflared dialing
# a port nothing was listening on yet (connection refused).
for _ in range(60):
    try:
        with socket.create_connection(("127.0.0.1", 8501), timeout=1):
            print("Streamlit is up on 127.0.0.1:8501")
            break
    except OSError:
        time.sleep(2)
else:
    print("Streamlit did not come up after 120s -- check the log below for a crash:")

!tail -n 40 /content/streamlit.log

In [ ]:
# Stays running -- prints "Your quick Tunnel has been created! Visit it at ...
# https://xxxx.trycloudflare.com". Open that URL directly, no password/interstitial
# step needed. Interrupt this cell to stop.
#
# Explicitly 127.0.0.1, not localhost -- on some Colab images "localhost" resolves
# to IPv6 [::1] first, which Streamlit doesn't bind by default, so cloudflared's
# dial fails with "connection refused" even though the app is up on IPv4.
!npx --yes cloudflared tunnel --url http://127.0.0.1:8501